# 2.6 Applications of Matrix Operations

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 2:** Matrices

---

### What you will learn

- How **matrix multiplication modulo 26** powers the **Hill cipher**, a classical cryptographic system.
- How to build a **modular matrix inverse** using the adjoint formula and the extended Euclidean algorithm.
- How to **encode** and **decode** messages, verifying the roundtrip property.
- How the **Leontief input–output model** uses $(I - C)^{-1}$ to determine production levels in a multi-sector economy.
- Why applications tie together every operation from this chapter: multiplication, inversion, determinants, and system solving.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose, inverse
from linalg_core.elimination import solve
from linalg_core.determinant import det_cofactor, cofactor_matrix, adjoint
from linalg_core import EPSILON

import numpy as np
import math

print("Setup complete.")

---

## 2. Motivation — Intercepted Cipher

An encrypted message has been intercepted:

$$\texttt{ciphertext} = [7,\; 0,\; 10,\; 6,\; 2,\; 2]$$

Intelligence tells us the message was encoded using a **Hill cipher** with the
$3 \times 3$ key matrix

$$K = \begin{bmatrix} 6 & 24 & 1 \\ 13 & 16 & 10 \\ 20 & 17 & 15 \end{bmatrix}$$

The Hill cipher maps letters $\texttt{A}=0, \texttt{B}=1, \ldots, \texttt{Z}=25$,
splits the plaintext into blocks of size $n$ (here $n=3$), and encrypts each block
by multiplying by $K$ modulo 26:

$$\mathbf{c} = K\mathbf{p} \pmod{26}$$

To **decode**, we need the modular inverse $K^{-1} \pmod{26}$ and compute

$$\mathbf{p} = K^{-1}\mathbf{c} \pmod{26}$$

This requires every tool from Chapter 2 — multiplication, determinants,
the adjoint formula, and modular arithmetic — all working together.
Let’s build the machinery and crack the code.

---

## 3. Build — Core Concepts

### 3.1 Hill Cipher Encoding

The **Hill cipher** (Lester Hill, 1929) is one of the first polygraphic substitution
ciphers. It operates on blocks of $n$ letters simultaneously using an
$n \times n$ **key matrix** $K$ whose entries are integers modulo 26.

**Encoding procedure:**

1. Map each letter to an integer: $\texttt{A}=0,\; \texttt{B}=1,\; \ldots,\; \texttt{Z}=25$.
2. Split the plaintext into column vectors of length $n$.
   If the message length is not a multiple of $n$, pad with $\texttt{X}$ (= 23).
3. For each plaintext block $\mathbf{p}$, compute the ciphertext block:

$$\mathbf{c} = K\mathbf{p} \pmod{26}$$

**Key requirement:** $\gcd(\det(K),\, 26) = 1$, so that $K$ is invertible modulo 26.

Let us implement the encoding step.

In [ ]:
def text_to_numbers(text):
    """Convert uppercase text to list of integers A=0 .. Z=25."""
    return [ord(ch) - ord('A') for ch in text.upper() if ch.isalpha()]


def numbers_to_text(nums):
    """Convert list of integers 0..25 back to uppercase text."""
    return ''.join(chr(int(n) % 26 + ord('A')) for n in nums)


def hill_encrypt(plaintext, K):
    """Encrypt plaintext using Hill cipher with key matrix K.

    K is an n x n Matrix.  Plaintext is padded with 'X' if needed.
    Returns (cipher_numbers, cipher_text).
    """
    n = K.rows
    nums = text_to_numbers(plaintext)

    while len(nums) % n != 0:
        nums.append(23)  # pad with X

    cipher = []
    for start in range(0, len(nums), n):
        block = Matrix.from_vector(nums[start:start + n])
        encrypted = multiply(K, block)
        for i in range(n):
            cipher.append(int(encrypted[i, 0]) % 26)

    return cipher, numbers_to_text(cipher)


K = Matrix.from_lists([
    [6, 24, 1],
    [13, 16, 10],
    [20, 17, 15],
])

print("Key matrix K:")
print(K)

plaintext = "ACT"
cipher_nums, cipher_text = hill_encrypt(plaintext, K)

print(f"\nPlaintext:    '{plaintext}'")
print(f"As numbers:   {text_to_numbers(plaintext)}")
print(f"Cipher nums:  {cipher_nums}")
print(f"Cipher text:  '{cipher_text}'")
print()
print("Step-by-step for 'ACT':")
p = text_to_numbers("ACT")
print(f"  p = {p}  (A=0, C=2, T=19)")
print(f"  K * p mod 26:")
for i in range(3):
    row = K.get_row(i)
    dot = sum(int(row[j]) * p[j] for j in range(3))
    print(f"    row {i}: {[int(x) for x in row]} . {p} = {dot}  mod 26 = {dot % 26}  ({chr(dot % 26 + 65)})")

### 3.2 Hill Cipher Decoding

To decode, we reverse the encryption by multiplying each ciphertext block
by $K^{-1} \pmod{26}$:

$$\mathbf{p} = K^{-1}\,\mathbf{c} \pmod{26}$$

**But this is not an ordinary inverse.** We need $K^{-1}$ where every operation
is performed modulo 26. The standard real-valued inverse from `linalg_core.ops.inverse`
won't work because we need integer arithmetic in $\mathbb{Z}_{26}$.

The **adjoint formula** gives us a path:

$$K^{-1} = \frac{1}{\det(K)} \cdot \text{adj}(K)$$

In modular arithmetic, "dividing by $\det(K)$" means multiplying by
the **modular multiplicative inverse** of $\det(K)$, i.e., the integer $d^{-1}$
such that

$$\det(K) \cdot d^{-1} \equiv 1 \pmod{26}$$

This inverse exists if and only if $\gcd(\det(K),\, 26) = 1$.

So the modular inverse is:

$$K^{-1} \equiv d^{-1} \cdot \text{adj}(K) \pmod{26}$$

### 3.3 Modular Inverse of a Matrix

We need two ingredients:

1. **Modular multiplicative inverse of an integer** — find $x$ such that $ax \equiv 1 \pmod{m}$.
   We use the **extended Euclidean algorithm**.
2. **Adjoint of $K$** — the transpose of the cofactor matrix, which we already have
   in `linalg_core.determinant.adjoint`.

Let us implement both and combine them.

In [ ]:
def extended_gcd(a, b):
    """Extended Euclidean algorithm.

    Returns (gcd, x, y) such that a*x + b*y = gcd(a, b).
    """
    if a == 0:
        return b, 0, 1
    g, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return g, x, y


def mod_multiplicative_inverse(a, m):
    """Find x such that a*x = 1 (mod m).

    Raises ValueError if gcd(a, m) != 1.
    """
    g, x, _ = extended_gcd(a % m, m)
    if g != 1:
        raise ValueError(
            f"Modular inverse does not exist: gcd({a}, {m}) = {g} != 1"
        )
    return x % m


def mod_inverse_matrix(K, mod=26):
    """Compute K^{-1} mod `mod` using the adjoint formula.

    K^{-1} = (det(K))^{-1} * adj(K)   (all mod `mod`)

    Returns a new Matrix with integer entries in [0, mod).
    """
    d = int(round(det_cofactor(K)))
    d_inv = mod_multiplicative_inverse(d, mod)

    adj_K = adjoint(K)

    n = K.rows
    result = Matrix(n, n)
    for i in range(n):
        for j in range(n):
            result[i, j] = (d_inv * int(round(adj_K[i, j]))) % mod
    return result


d = int(round(det_cofactor(K)))
print(f"det(K) = {d}")
print(f"det(K) mod 26 = {d % 26}")
print(f"gcd(det(K), 26) = {math.gcd(abs(d) % 26, 26)}")
print()

d_inv = mod_multiplicative_inverse(d, 26)
print(f"Modular inverse of {d % 26} mod 26 = {d_inv}")
print(f"Verification: {d % 26} * {d_inv} = {(d % 26) * d_inv} mod 26 = {((d % 26) * d_inv) % 26}")
print()

print("Adjoint of K:")
print(adjoint(K))
print()

K_inv = mod_inverse_matrix(K, 26)
print("K^{-1} mod 26:")
print(K_inv)

### 3.4 Encode / Decode Demo

Now we have all the pieces. Let us:

1. Encode a message ("HELP") with the key $K$.
2. Decode it back using $K^{-1} \pmod{26}$.
3. Verify the **roundtrip property**: decode(encode(plaintext)) = plaintext.
4. **Crack the intercepted message** from the motivation.

In [ ]:
def hill_decrypt(cipher_nums, K_inv_mod, mod=26):
    """Decrypt a list of cipher numbers using K^{-1} mod `mod`.

    Returns (plain_numbers, plain_text).
    """
    n = K_inv_mod.rows
    plain = []
    for start in range(0, len(cipher_nums), n):
        block = Matrix.from_vector(cipher_nums[start:start + n])
        decrypted = multiply(K_inv_mod, block)
        for i in range(n):
            plain.append(int(round(decrypted[i, 0])) % mod)
    return plain, numbers_to_text(plain)


print("=" * 60)
print("ENCODE / DECODE DEMO: 'HELPXX'")
print("=" * 60)

message = "HELPXX"
print(f"\nOriginal message:  '{message}'")
print(f"As numbers:        {text_to_numbers(message)}")

enc_nums, enc_text = hill_encrypt(message, K)
print(f"\nEncrypted numbers: {enc_nums}")
print(f"Encrypted text:    '{enc_text}'")

dec_nums, dec_text = hill_decrypt(enc_nums, K_inv, 26)
print(f"\nDecrypted numbers: {dec_nums}")
print(f"Decrypted text:    '{dec_text}'")

roundtrip_ok = text_to_numbers(message) == dec_nums
print(f"\nRoundtrip verified: {roundtrip_ok}")

print()
print("=" * 60)
print("CRACK THE INTERCEPTED MESSAGE")
print("=" * 60)

intercepted = [7, 0, 10, 6, 2, 2]
print(f"\nIntercepted ciphertext: {intercepted}")

plain_nums, plain_text = hill_decrypt(intercepted, K_inv, 26)
print(f"Decoded numbers:        {plain_nums}")
print(f"Decoded message:        '{plain_text}'")
print()

verify_enc, _ = hill_encrypt(plain_text, K)
print(f"Re-encrypt '{plain_text}': {verify_enc}")
print(f"Matches intercepted?     {verify_enc == intercepted}")

### 3.5 Leontief Input–Output Model

The **Leontief input–output model** (Wassily Leontief, Nobel Prize 1973)
describes an economy with $n$ interdependent sectors. Each sector both
produces goods and consumes goods from other sectors.

**Setup:**

- $\mathbf{x}$ = total output vector (what each sector produces).
- $C$ = **consumption matrix** where $c_{ij}$ is the fraction of sector $j$’s
  output consumed by sector $i$.
- $\mathbf{d}$ = **external demand** vector.

The economy must satisfy:

$$\mathbf{x} = C\mathbf{x} + \mathbf{d}$$

Total output = internal consumption + external demand.

Rearranging:

$$(I - C)\mathbf{x} = \mathbf{d}$$

If $I - C$ is invertible (which it is when $C$ is a valid consumption matrix
with column sums less than 1), then:

$$\mathbf{x} = (I - C)^{-1}\mathbf{d}$$

The matrix $(I - C)^{-1}$ is called the **Leontief inverse**. Its $(i,j)$ entry
tells us how much sector $i$’s output must increase for each unit increase
in sector $j$’s external demand.

In [ ]:
def leontief_solve(C, d):
    """Solve (I - C)x = d for the Leontief input-output model.

    C: n x n consumption matrix (Matrix)
    d: external demand as a list of n floats

    Returns (x, I_minus_C) where x is the total output vector.
    """
    n = C.rows
    I_n = Matrix.identity(n)
    I_minus_C = add(I_n, scalar_mul(C, -1))

    aug = Matrix(n, n + 1)
    for i in range(n):
        for j in range(n):
            aug[i, j] = I_minus_C[i, j]
        aug[i, n] = d[i]

    sol_type, result = solve(aug)
    if sol_type != "unique":
        raise ValueError(f"System does not have a unique solution: {sol_type}")

    return result, I_minus_C


print("Leontief solver ready.")

### 3.6 Demo — 3-Sector Economy

Consider a simplified economy with three sectors:

| Sector | Description |
|---|---|
| **Agriculture** (sector 1) | Produces food and raw materials |
| **Manufacturing** (sector 2) | Produces finished goods |
| **Services** (sector 3) | Provides transportation, retail, etc. |

The **consumption matrix** $C$ describes internal dependencies:

$$C = \begin{bmatrix}
0.2 & 0.3 & 0.1 \\
0.1 & 0.1 & 0.3 \\
0.3 & 0.2 & 0.2
\end{bmatrix}$$

For example, $c_{12} = 0.3$ means that Agriculture consumes 30% of
Manufacturing’s total output.

External demand is $\mathbf{d} = (50, 30, 20)^T$ (in billions of dollars).

**Question:** What total output $\mathbf{x}$ must each sector produce to meet
both internal consumption and external demand?

In [ ]:
C = Matrix.from_lists([
    [0.2, 0.3, 0.1],
    [0.1, 0.1, 0.3],
    [0.3, 0.2, 0.2],
])

d = [50, 30, 20]

print("Consumption matrix C:")
print(C)
print(f"\nExternal demand d = {d}")
print()

col_sums = [sum(C[i, j] for i in range(3)) for j in range(3)]
print("Column sums of C:", [round(s, 2) for s in col_sums])
print("All < 1 (productive economy)?  ", all(s < 1.0 for s in col_sums))
print()

x, I_minus_C = leontief_solve(C, d)

print("I - C:")
print(I_minus_C)
print()

sector_names = ["Agriculture", "Manufacturing", "Services"]
print("Solution --- Total output required:")
for i, name in enumerate(sector_names):
    print(f"  {name:15s}: ${x[i]:8.2f} billion")
print()

print("Interpretation:")
print(f"  To meet $50B external demand for Agriculture,")
print(f"  $30B for Manufacturing, and $20B for Services,")
print(f"  Agriculture must produce ${x[0]:.2f}B total (the extra")
print(f"  ${x[0] - d[0]:.2f}B is consumed internally by other sectors).")
print()

print("Verification: (I - C)x - d should be near zero:")
x_vec = Matrix.from_vector(x)
Ax = multiply(I_minus_C, x_vec)
residual = [Ax[i, 0] - d[i] for i in range(3)]
print(f"  residual = {[f'{r:.2e}' for r in residual]}")
print(f"  max |residual| = {max(abs(r) for r in residual):.2e}")
print(f"  < EPSILON ({EPSILON})? {max(abs(r) for r in residual) < EPSILON}")

---

## 4. Verify — Cross-Check with NumPy

We run three verification suites:

1. **Hill cipher roundtrip** — encode then decode yields the original message.
2. **Modular inverse identity** — $K \cdot K^{-1} \equiv I \pmod{26}$.
3. **Leontief residual** — $(I - C)\mathbf{x} - \mathbf{d}$ is within machine epsilon.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())


print("=" * 60)
print("VERIFICATION 1: Hill Cipher Roundtrip")
print("=" * 60)

test_messages = ["ACT", "HELP", "MATRIX", "LINEAR", "ATTACK", "CRYPTO"]
all_pass = True
for msg in test_messages:
    enc_nums, enc_text = hill_encrypt(msg, K)
    dec_nums, dec_text = hill_decrypt(enc_nums, K_inv, 26)

    padded = msg
    while len(padded) % 3 != 0:
        padded += "X"

    ok = dec_text == padded
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] '{msg}' -> {enc_nums} -> '{dec_text}'")
    all_pass = all_pass and ok

print(f"\n  Overall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 2: K * K^{-1} mod 26 = I")
print("=" * 60)

product = multiply(K, K_inv)
n = K.rows
I_mod = Matrix(n, n)
for i in range(n):
    for j in range(n):
        I_mod[i, j] = int(round(product[i, j])) % 26

print("\nK * K^{-1} (raw):")
print(product)
print("\nK * K^{-1} mod 26:")
print(I_mod)

I_3 = Matrix.identity(3)
is_identity = I_mod == I_3
print(f"\nEquals I_3? {is_identity}")

K_np = to_np(K).astype(int)
K_inv_np = to_np(K_inv).astype(int)
product_np = (K_np @ K_inv_np) % 26
np_is_identity = np.array_equal(product_np, np.eye(3, dtype=int))
print(f"NumPy cross-check: {np_is_identity}")

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Leontief Residual")
print("=" * 60)

C_np = to_np(C)
d_np = np.array(d, dtype=float)
I_minus_C_np = np.eye(3) - C_np
x_np = np.linalg.solve(I_minus_C_np, d_np)

print("\nOur solution:  ", [round(v, 4) for v in x])
print("NumPy solution:", [round(v, 4) for v in x_np])

diff = [abs(x[i] - x_np[i]) for i in range(3)]
print(f"\nMax difference:  {max(diff):.2e}")
print(f"Below EPSILON?   {max(diff) < EPSILON}")

residual_np = I_minus_C_np @ np.array(x) - d_np
print(f"\nNumPy residual:  {[f'{r:.2e}' for r in residual_np]}")
print(f"Max |residual|:  {np.max(np.abs(residual_np)):.2e}")

print(f"\n[{'PASS' if max(diff) < EPSILON else 'FAIL'}] "
      f"Leontief solution matches NumPy")

---

## 5. Exercises

Test your understanding with the following problems.

### Exercise 1 (Easy)

Use the $2 \times 2$ key matrix

$$K_2 = \begin{bmatrix} 3 & 5 \\ 1 & 2 \end{bmatrix}$$

to encode the message **"CODE"** using the Hill cipher.

Then compute $K_2^{-1} \pmod{26}$ and decode your ciphertext.
Verify the roundtrip.

*Hint:* $\det(K_2) = 3 \cdot 2 - 5 \cdot 1 = 1$, so the modular inverse
of the determinant is simply 1. The adjoint of a $2 \times 2$ matrix
$\begin{bmatrix} a & b \\ c & d \end{bmatrix}$ is
$\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

A 3-sector economy has the consumption matrix

$$C = \begin{bmatrix}
0.15 & 0.25 & 0.10 \\
0.20 & 0.05 & 0.30 \\
0.10 & 0.30 & 0.15
\end{bmatrix}$$

and external demand $\mathbf{d} = (80,\; 60,\; 40)^T$.

1. Verify that $C$ is a valid consumption matrix (all column sums < 1).
2. Solve $(I - C)\mathbf{x} = \mathbf{d}$ using `leontief_solve`.
3. Compute the **Leontief inverse** $(I - C)^{-1}$ using `inverse` from
   `linalg_core.ops`.
4. Verify that $(I - C)^{-1}\mathbf{d} = \mathbf{x}$.
5. **Interpretation:** If external demand for sector 2 increases by \$10B
   (from 60 to 70), how much additional output does each sector need?
   (Answer: look at column 2 of $(I - C)^{-1}$ and multiply by 10.)

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) — Known-Plaintext Attack

In a **known-plaintext attack**, you have both the plaintext and ciphertext
for some messages, and you want to recover the unknown key matrix.

Suppose you know that the $3 \times 3$ key $K$ encrypted the plaintext
blocks:

| Plaintext block | Ciphertext block |
|---|---|
| (0, 2, 19) = "ACT" | (15, 14, 7) = "POH" |
| (18, 19, 17) = "STR" | (9, 6, 2) = "JGC" |
| (8, 10, 4) = "IKE" | (6, 18, 0) = "GSA" |

Since $K \cdot P = C \pmod{26}$ where $P$ and $C$ are the $3 \times 3$
matrices of column blocks, we can recover $K = C \cdot P^{-1} \pmod{26}$.

1. Form the $3 \times 3$ plaintext matrix $P$ (columns are plaintext blocks).
2. Compute $P^{-1} \pmod{26}$ using `mod_inverse_matrix`.
3. Compute $K = C \cdot P^{-1} \pmod{26}$.
4. Verify that the recovered $K$ matches the key matrix from this notebook.

*Hint:* Be careful with the matrix orientation. The plaintext blocks are
**columns** of $P$, so $P$ = `Matrix.from_lists([[0,18,8],[2,19,10],[19,17,4]])`.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Application | Key Matrix Concept | Formula |
|---|---|---|
| **Hill cipher (encode)** | Matrix–vector multiplication mod 26 | $\mathbf{c} = K\mathbf{p} \pmod{26}$ |
| **Hill cipher (decode)** | Modular matrix inverse via adjoint | $\mathbf{p} = K^{-1}\mathbf{c} \pmod{26}$ |
| **Modular inverse** | Determinant + adjoint + extended GCD | $K^{-1} \equiv (\det K)^{-1} \cdot \text{adj}(K) \pmod{26}$ |
| **Known-plaintext attack** | Inverse of plaintext matrix | $K = C \cdot P^{-1} \pmod{26}$ |
| **Leontief model** | System solving / matrix inverse | $\mathbf{x} = (I - C)^{-1}\mathbf{d}$ |

**Takeaway:** The abstract operations from earlier sections — multiplication,
transposition, determinants, cofactors, adjoints, and inversion — combine
to solve real problems in cryptography and economics. The Hill cipher
shows that even **modular** arithmetic follows the same structural rules,
and the Leontief model demonstrates that understanding $(I - C)^{-1}$
has direct economic meaning.